# 00 — Setup & Smoke Test

**Goal**: verify environment end-to-end before running any real experiment.

Checks:
1. `AI_GATEWAY_API_KEY` is set
2. Each classifier candidate + judge model returns a valid Pydantic object
3. Disk cache works (second call hits cache, sub-10ms)
4. All stratified datasets are present with expected class balance

If any check fails, stop here and fix before continuing.

In [1]:
import sys
sys.path.insert(0, '..')

import asyncio
import time
from lib.config import CLASSIFIER_CANDIDATES, JUDGE, gateway_key
from lib.gateway import classify_json
from lib.schemas import StarsOnly, JudgeScore
from lib.prompts import CLASSIFIER_SYSTEM, zero_shot, JUDGE_SYSTEM, judge as judge_prompt
from lib.datasets import read_jsonl, class_dist

print('env:', 'ok' if gateway_key() else 'MISSING')
print('classifier candidates:', CLASSIFIER_CANDIDATES)
print('judge:', JUDGE)

env: ok
classifier candidates: ['openai/gpt-5-nano', 'deepseek/deepseek-v3.2', 'google/gemini-2.0-flash']
judge: anthropic/claude-haiku-4.5


## Classifier smoke — each candidate on the same review

Same input → compare latency, output, cost footprint. This is a sanity check, not an evaluation.

In [2]:
REVIEW = "Best ramen I've had in years. Broth rich, noodles perfect, staff friendly. Wait was 25 min but worth it."

results = []
for m in CLASSIFIER_CANDIDATES:
    t0 = time.time()
    obj, meta = await classify_json(StarsOnly, CLASSIFIER_SYSTEM, zero_shot(REVIEW), model=m)
    ms = int((time.time() - t0) * 1000)
    results.append({'model': m, 'stars': obj.stars if obj else None, 'explanation': (obj.explanation[:80] + '...') if obj else None, 'ms': ms, 'cached': meta.get('cached')})

import pandas as pd
pd.DataFrame(results)

,model,stars,explanation,ms,cached
0,openai/gpt-5-nano,5,"Excellent ramen; broth rich, noodles perfect, ...",3551,False
1,deepseek/deepseek-v3.2,5,The review expresses extremely positive sentim...,3429,False
2,google/gemini-2.0-flash,5,"The review is overwhelmingly positive, citing ...",1418,False


In [3]:
t0 = time.time()
obj, meta = await classify_json(StarsOnly, CLASSIFIER_SYSTEM, zero_shot(REVIEW))
print(f'2nd call cached={meta.get("cached")} latency={int((time.time()-t0)*1000)}ms')

2nd call cached=True latency=0ms


## Judge smoke — Haiku 4.5

Feed a fake business-response triple and confirm the judge parses into `JudgeScore`.

In [4]:
judge_p = judge_prompt(
    review=REVIEW,
    stars=5,
    signal_type='compliment',
    signal_text='rich broth, perfect noodles',
    response='Thank you so much for the kind words! We take huge pride in our broth — happy you noticed. Looking forward to serving you again.',
)

score, meta = await classify_json(JudgeScore, JUDGE_SYSTEM, judge_p, model=JUDGE)
print('judge model:', meta.get('model'))
print('score:', score)

judge model: anthropic/claude-haiku-4.5
score: faithfulness=4 politeness=5 actionability=2 rationale="The response accurately captures the broth and noodles compliments but omits mention of the friendly staff and the 25-minute wait, which were notable review elements. The tone is warm and professional. However, actionability is weak—there's no concrete next step, offer, or invitation beyond a generic 'looking forward to serving you again.'"


## Dataset integrity

All five JSONL files should exist with the expected stratification.

In [5]:
files = {
    'yelp_eval':      ('data/yelp_eval.jsonl',      {1: 100, 2: 100, 3: 100, 4: 100, 5: 100}),
    'yelp_fewshot':   ('data/yelp_fewshot.jsonl',   {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}),
    'amazon_eval':    ('data/amazon_eval.jsonl',    {1: 60, 2: 60, 3: 60, 4: 60, 5: 60}),
    'imdb_eval':      ('data/imdb_eval.jsonl',      {1: 150, 5: 150}),
    'adversarial':    ('data/adversarial.jsonl',    None),
}

for name, (path, expected) in files.items():
    rows = read_jsonl('../' + path)
    dist = class_dist(rows)
    ok = '✅' if (expected is None or dist == expected) else '❌'
    print(f'{ok} {name:15s} n={len(rows):5d}  dist={dist}')

✅ yelp_eval       n=  500  dist={1: 100, 2: 100, 3: 100, 4: 100, 5: 100}
✅ yelp_fewshot    n=    5  dist={1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
✅ amazon_eval     n=  300  dist={1: 60, 2: 60, 3: 60, 4: 60, 5: 60}
✅ imdb_eval       n=  300  dist={1: 150, 5: 150}
✅ adversarial     n=  160  dist={1: 16, 2: 24, 3: 40, 4: 40, 5: 40}


If all rows print ✅ and each classifier returned a valid `stars` integer, the stack is ready. Proceed to `01_json_prompt.ipynb`.